# 06. HSMM follower: переходы с учётом длительностей

`ScoreFollowerHSMM` — это «улучшенный HMM», в котором переходные вероятности **пересчитываются на каждом событии** в зависимости от того, сколько времени уже прошло в текущем state.

Четыре исхода (всегда сумма = 1):

- **stay** — остаться в текущем state (короче номинала → высокий);
- **advance** — перейти на следующий state (близко к номиналу → высокий);
- **skip** — перепрыгнуть через одно состояние (страховка от пропуска ноты);
- **leap** — пропустить два состояния (структурная ошибка).

Точная формула — кусочно-линейная по `ratio = elapsed / nominal_duration`. См. метод `_transition_probabilities` в `hsmm_follower.py`.

В этом ноутбуке:

1. Прогоняем HSMM на `rubato` и `noisy`, сохраняем α и `last_transition_probabilities` на каждом шаге.
2. Визуализируем как меняются `stay/advance/skip/leap` по времени.
3. Сравниваем результат с HMM из ноутбука 05.

**Важно**: текущий код — это **эвристика**. Честный HSMM по Cont 2010 (явные `p_i(d)` log-normal) — следующий шаг, см. трек 2A в `analysis.md`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt

from musictech.core import ScoreFollowerHSMM
from dataset_viewer import load_performance

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
DATA = PROJECT_ROOT / "generated_dataset"

In [ ]:
def run_hsmm(score_path: Path, perf_path: Path, sigma: float = 2.5):
    follower = ScoreFollowerHSMM(str(score_path), sigma=sigma)
    performance = load_performance(perf_path)
    alphas, predicted, transitions = [], [], []
    for event in performance:
        idx = follower.process_event(event["pitch"], event["timestamp"])
        alphas.append(follower.alpha.copy())
        predicted.append(idx)
        transitions.append(dict(follower.last_transition_probabilities))
    return follower, performance, np.array(alphas), predicted, transitions

## 1. Прогон на `rubato`

In [ ]:
follower, performance, alphas, predicted, transitions = run_hsmm(
    DATA / "rubato.json", DATA / "rubato.mid"
)
print("predicted:", predicted)
print("expected :", list(range(follower.N)))

## 2. Как меняются переходные вероятности во времени

Когда исполнитель играет коротко (быстрее номинала), `stay` высокая. Когда исполнитель медленный — `advance` высокая. На rubato это должно «дышать».

In [ ]:
names = ["stay", "advance", "skip", "leap"]
transitions_arr = np.array([[t[n] for n in names] for t in transitions])

fig, ax = plt.subplots()
for i, name in enumerate(names):
    ax.plot(transitions_arr[:, i], marker="o", label=name)
ax.set_xlabel("event #")
ax.set_ylabel("probability")
ax.set_title("HSMM transition probabilities over time ('rubato')")
ax.legend()
plt.show()

## 3. Тепловая карта α

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(alphas, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
ax.plot(predicted, range(len(predicted)), color="white", marker="o", linestyle="")
ax.set_xlabel("score state index")
ax.set_ylabel("event #")
ax.set_title("HSMM α on 'rubato'")
plt.colorbar(im, ax=ax, label="α(state)")
plt.tight_layout()
plt.show()

## 4. На шуме: что меняется по сравнению с HMM

Шумовые события HSMM игнорирует чуть охотнее, чем HMM — благодаря тому, что `stay` высокая когда `elapsed < nominal`. Лишняя нота не двигает позицию.

In [ ]:
follower, performance, alphas, predicted, transitions = run_hsmm(
    DATA / "noisy.json", DATA / "noisy.mid"
)
print("perf pitches  :", [int(e["pitch"]) for e in performance])
print("predicted idx :", predicted)

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(alphas, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
ax.plot(predicted, range(len(predicted)), color="white", marker="o", linestyle="")
ax.set_xlabel("score state")
ax.set_ylabel("event #")
ax.set_title("HSMM α on 'noisy'")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5. Что важно для тезисов и куда дальше

- В тезисах HSMM — **базовый трекер**, на чьём `α_t` будет работать RL. Текущая реализация — рабочая эвристика, но не «честный HSMM».
- Для академически чистого HSMM нужно (трек 2A из `analysis.md`):
  - явные распределения длительности `p_i(d)` (log-normal с медианой = `nominal_duration`);
  - beam-search forward для крупных пьес (N ≈ 3800);
  - структурные переходы по Nakamura 2015.
- Метрики качества HSMM (alignment accuracy, onset-error, recovery latency) пока считаются вручную из `autoplay_offset_benchmark.py`. Их вынос в `musictech/evaluation/` — следующий блокер.